# Infra-FM: AlphaEarth Foundations (2024) — Linear Probe

Fourth FM in the Infra-Bench cross-FM comparison, alongside SatlasPretrain S2,
SatlasPretrain S1, CROMA, and Prithvi-EO-2.0.

Unlike the other FMs, **AlphaEarth exposes precomputed embeddings**, not a
trainable encoder. The "frozen" / "fine-tune" distinction does not apply —
embeddings are always frozen by design.

## What this notebook does

1. **Stage 1** (one-time, ~hours depending on EE quota): fetches 2024
   `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL` region-mean embeddings (64 floats
   per tile) for every v1 multi-sector tile, writes to a Drive parquet.
   Resumable — re-runs skip asset_ids already in the parquet.
2. **Stage 2**: trains a single Linear layer (64 -> 13) on the embeddings,
   using the same 70/15/15 stratified split (seed=42) as the other FMs.

## Per-sector F1 — corrected v2 definition (read this)

`per_sector` here is the **macro-average of the per-class F1s** for the
classes in that sector, computed on the FULL test set. NOT a macro-F1
restricted to in-sector samples (which is what some earlier notebooks
produced and which we've since deprecated for being misleading on imbalanced
sectors). The results JSON saves the same value into both `per_sector` and
`per_sector_corrected` plus a `per_sector_correction_note` describing the
definition, so downstream cross-FM synthesis code that reads
`per_sector_corrected` works unchanged.

## Spec correction worth flagging

The build spec said the CROMA notebook uses *"deterministic split by
asset_id hash, 75/15/10"*. The CROMA notebook actually uses
`stratified_split` by class at 70/15/15 with `SEED=42` (same as Prithvi,
SatlasS1, and SatlasS2). Same-split-across-FMs is the fairness invariant,
so this notebook uses the real CROMA logic (70/15/15 stratified, seed=42).

## Outputs

- `/.../data/alphaearth/embeddings_2024.parquet`
  Columns: `asset_id`, `region`, `sector`, `asset_type`, `A00`..`A63`.
- `/.../results/fm_eval_alphaearth_v1/alphaearth_v1_linear_probe_results.json`
- `/.../results/fm_eval_alphaearth_v1/confusion_matrix_alphaearth_7region.png`

The smoke check (cell 11) defaults to `SMOKE_ONLY=True`. The fetch stage
(cell 7) always runs — it's idempotent and bounded.


In [7]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}  (not required — 64-D linear probe is CPU-fine)')


Mounted at /content/drive
GPU available: True  (not required — 64-D linear probe is CPU-fine)


In [8]:
%%capture
!pip install -q earthengine-api pyarrow pandas matplotlib scikit-learn


In [9]:
# Idempotent Earth Engine auth. Try Initialize first — if it works, the
# notebook session already has cached credentials. Only call Authenticate
# if Initialize fails, otherwise re-runs hang waiting for browser confirm.
import ee

EE_PROJECT = 'infrabench'   # change if your GCP project differs

try:
    ee.Initialize(project=EE_PROJECT)
    print(f'EE initialized with cached credentials (project={EE_PROJECT}).')
except Exception:
    print('No cached EE credentials — authenticating...')
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project=EE_PROJECT)
    print(f'EE initialized (project={EE_PROJECT}).')

# Sanity probe — confirm the AlphaEarth collection is reachable.
ae_collection = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
print(f'AlphaEarth ImageCollection accessible: {ae_collection.size().getInfo()} images total')


EE initialized with cached credentials (project=infrabench).
AlphaEarth ImageCollection accessible: 97155 images total


In [10]:
# Optional code-zip extraction — same convention as CROMA / Prithvi notebooks.
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/downstream').exists() and not Path(f'{EXTRACT_TO}/infra_fm_code_only/downstream').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')
    else:
        print(f'No code zip at {CODE_ZIP} — notebook is self-contained, OK to skip.')
else:
    print(f'Code already at {EXTRACT_TO}.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)


Extracting /content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip -> /content/infrabench_repo ...
done.


In [11]:
import os
from pathlib import Path

# ---- Drive paths -----------------------------------------------------------
DATASETS_DRIVE       = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL       = '/content/datasets'
ALPHAEARTH_DATA_DIR  = f'{DRIVE_ROOT}/data/alphaearth'
EMBEDDINGS_PARQUET   = f'{ALPHAEARTH_DATA_DIR}/embeddings_2024.parquet'
OUTPUT_DIR           = f'{DRIVE_ROOT}/results/fm_eval_alphaearth_v1'
os.makedirs(DATASETS_LOCAL, exist_ok=True)
os.makedirs(ALPHAEARTH_DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Coverage --------------------------------------------------------------
REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

# ---- 13-class taxonomy (locked, must match other FM notebooks) -------------
CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',          # pools substation_untyped + substation_minor
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.treatment.plant',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX        = {n: i for i, n in enumerate(CLASS_NAMES)}
CLASS_IDX_TO_SECTOR = {i: n.split('.')[0] for i, n in enumerate(CLASS_NAMES)}

ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':                  'water.treatment.plant',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

# ---- AlphaEarth ------------------------------------------------------------
AE_COLLECTION_ID = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
AE_YEAR_START    = '2024-01-01'
AE_YEAR_END      = '2025-01-01'
AE_BAND_COUNT    = 64
AE_BAND_NAMES    = [f'A{i:02d}' for i in range(AE_BAND_COUNT)]
AE_SCALE_M       = 10              # AlphaEarth native resolution
AE_BATCH_SIZE    = 500             # tested safe size for reduceRegions

# ---- Training config (mirrors CROMA / Prithvi / SatlasS1) -----------------
SEED       = 42
LP_EPOCHS  = 25
LP_BATCH   = 16
LP_LR      = 1e-3
WEIGHT_CAP = 10.0

# ---- Determinism -----------------------------------------------------------
def set_seed(seed: int) -> None:
    import random
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

print(f'Regions ({len(REGIONS)}): {REGIONS}')
print(f'Classes ({len(CLASS_NAMES)})')
print(f'AlphaEarth collection:  {AE_COLLECTION_ID}')
print(f'  year window:          [{AE_YEAR_START}, {AE_YEAR_END})')
print(f'  bands:                {AE_BAND_NAMES[0]}..{AE_BAND_NAMES[-1]}  (64)')
print(f'  reduceRegions batch:  {AE_BATCH_SIZE}')
print(f'Linear probe:           {LP_EPOCHS} epochs, batch {LP_BATCH}, lr {LP_LR}')
print(f'Embedding parquet:      {EMBEDDINGS_PARQUET}')
print(f'Output dir:             {OUTPUT_DIR}')


Regions (7): ['africa', 'asia', 'australia-oceania', 'central-america', 'europe', 'north-america', 'south-america']
Classes (13)
AlphaEarth collection:  GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL
  year window:          [2024-01-01, 2025-01-01)
  bands:                A00..A63  (64)
  reduceRegions batch:  500
Linear probe:           25 epochs, batch 16, lr 0.001
Embedding parquet:      /content/drive/MyDrive/infra_fm/data/alphaearth/embeddings_2024.parquet
Output dir:             /content/drive/MyDrive/infra_fm/results/fm_eval_alphaearth_v1


In [12]:
import json, re, shutil, zipfile, time
import pandas as pd
from pathlib import Path

# We only need bboxes from each cell's manifest.json — no images on local disk.
# The fetch operates entirely on the Drive-side manifest data.

MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')
drive_path = Path(DATASETS_DRIVE)


def manifest_records_from_drive():
    """Yield (region, sector, manifest_dict) for every v1_1k cell on Drive.
    Handles both folder-on-Drive and zip-on-Drive sources without extracting
    full image folders — we only need manifest.json."""
    if not drive_path.exists():
        raise RuntimeError(f'{DATASETS_DRIVE} not found — is Drive mounted?')
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m:
            continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS:
            continue
        # Folder source: read manifest directly.
        if entry.is_dir():
            mp = entry / 'manifest.json'
            if not mp.exists():
                print(f'  [skip] {region}/{sector}: no manifest.json in folder')
                continue
            with mp.open() as f:
                yield region, sector, json.load(f)
            continue
        # Zip source: extract manifest.json only.
        if entry.suffix == '.zip':
            folder_name = f'dataset_{region}_{sector}_v1_1k'
            with zipfile.ZipFile(entry) as zf:
                # Try wrapped and unwrapped layouts
                candidates = [f'{folder_name}/manifest.json', 'manifest.json']
                manifest_bytes = None
                for cand in candidates:
                    try:
                        manifest_bytes = zf.read(cand)
                        break
                    except KeyError:
                        continue
                if manifest_bytes is None:
                    print(f'  [skip] {region}/{sector}: no manifest.json in zip')
                    continue
            yield region, sector, json.loads(manifest_bytes.decode('utf-8'))


records = []
n_skipped_bad_label = 0
n_skipped_bad_bbox  = 0

for region, sector, manifest in manifest_records_from_drive():
    cell_records = manifest.get('records', [])
    print(f'  {region:<22s} {sector:<10s} {len(cell_records):>6d} records')
    for r in cell_records:
        at = r.get('asset_type')
        if at not in ASSET_TYPE_MAP:
            n_skipped_bad_label += 1
            continue
        bbox = r.get('bbox')
        if not bbox or len(bbox) != 4:
            n_skipped_bad_bbox += 1
            continue
        # Manifests carry bbox = [west, south, east, north]. Some records may
        # also carry centroid (lat, lon); we deliberately use the full bbox
        # per spec ("region-mean over the manifest bbox, NOT centroid-only").
        west, south, east, north = bbox
        records.append({
            'asset_id':   r.get('asset_id', ''),
            'region':     region,
            'sector':     sector,
            'asset_type': ASSET_TYPE_MAP[at],
            'west':       float(west),
            'south':      float(south),
            'east':       float(east),
            'north':      float(north),
        })

assets_df = pd.DataFrame(records).drop_duplicates(subset=['asset_id'])
print()
print(f'Built asset list: {len(assets_df):,} unique assets across '
      f'{assets_df["region"].nunique()} regions, {assets_df["sector"].nunique()} sectors')
print(f'  skipped (asset_type not in ASSET_TYPE_MAP): {n_skipped_bad_label:,}')
print(f'  skipped (bbox missing/malformed):           {n_skipped_bad_bbox:,}')
print()
print('Per-region counts:')
print(assets_df.groupby('region').size().to_string())
print()
print('Per-class counts:')
print(assets_df.groupby('asset_type').size().sort_values(ascending=False).to_string())

# Sanity check vs spec expectation.
n = len(assets_df)
if not (16000 <= n <= 22000):
    print(f'\nNOTE: asset count {n:,} is outside the expected 18-19k range — '
          f'investigate before launching the fetch.')


  africa                 energy        949 records
  africa                 telecom         1 records
  africa                 transport    1000 records
  africa                 water         892 records
  asia                   energy        889 records
  asia                   telecom        28 records
  asia                   transport     891 records
  asia                   water         958 records
  australia-oceania      energy       1002 records
  australia-oceania      telecom        12 records
  australia-oceania      transport    1000 records
  australia-oceania      water        1000 records
  central-america        energy        999 records
  central-america        telecom         1 records
  central-america        transport     566 records
  central-america        water         999 records
  europe                 energy        412 records
  europe                 telecom       257 records
  europe                 transport     577 records
  europe                 water 

In [13]:
# Stage 1: fetch AlphaEarth 2024 region-mean embeddings via reduceRegions.
# Idempotent — re-runs skip asset_ids already in the parquet.

import time
import pandas as pd
import numpy as np
import ee

# ---- AlphaEarth image: mosaic of 2024 annual embeddings -------------------
ae_image = (ee.ImageCollection(AE_COLLECTION_ID)
            .filterDate(AE_YEAR_START, AE_YEAR_END)
            .mosaic())
ae_band_names = ae_image.bandNames().getInfo()
print(f'AlphaEarth 2024 mosaic has {len(ae_band_names)} bands: {ae_band_names[:6]}...')
assert len(ae_band_names) == AE_BAND_COUNT, \
    f'AlphaEarth band count {len(ae_band_names)} != expected {AE_BAND_COUNT}'

# ---- Load existing parquet (resume support) -------------------------------
existing = None
done_ids = set()
parquet_path = Path(EMBEDDINGS_PARQUET)
if parquet_path.exists():
    existing = pd.read_parquet(parquet_path)
    done_ids = set(existing['asset_id'].astype(str))
    print(f'Resume: found existing parquet with {len(done_ids):,} asset_ids done')
else:
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

todo_df = assets_df[~assets_df['asset_id'].astype(str).isin(done_ids)].reset_index(drop=True)
print(f'To fetch: {len(todo_df):,}  (skipping {len(done_ids):,} already done)')

# ---- Batch fetch helper ---------------------------------------------------
def _fc_from_batch(batch_rows):
    """Build an ee.FeatureCollection from a slice of `todo_df`."""
    features = []
    for _, r in batch_rows.iterrows():
        geom = ee.Geometry.Rectangle([r['west'], r['south'], r['east'], r['north']])
        features.append(ee.Feature(geom, {'asset_id': str(r['asset_id'])}))
    return ee.FeatureCollection(features)


def _fetch_batch(batch_rows):
    """Run reduceRegions on one batch and return a DataFrame of results."""
    fc = _fc_from_batch(batch_rows)
    reduced = ae_image.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean(),
        scale=AE_SCALE_M,
    )
    # Pull the result client-side. .getInfo() blocks until EE finishes.
    info = reduced.getInfo()
    rows = []
    for feat in info.get('features', []):
        props = feat.get('properties', {})
        asset_id = props.get('asset_id')
        row = {'asset_id': asset_id}
        for b in AE_BAND_NAMES:
            row[b] = props.get(b)
        rows.append(row)
    return pd.DataFrame(rows)


# ---- Loop with logging + error tolerance ----------------------------------
fetched_chunks = []
n_total = len(todo_df)
n_done = 0
n_batches = (n_total + AE_BATCH_SIZE - 1) // AE_BATCH_SIZE
t_start = time.time()
errors_log = []

for batch_idx in range(n_batches):
    lo = batch_idx * AE_BATCH_SIZE
    hi = min(lo + AE_BATCH_SIZE, n_total)
    batch = todo_df.iloc[lo:hi]
    t0 = time.time()
    try:
        result = _fetch_batch(batch)
    except Exception as e:
        msg = f'batch {batch_idx+1}/{n_batches} ({lo}..{hi}) error: {e.__class__.__name__}: {e}'
        print(f'  [ERROR] {msg}')
        errors_log.append(msg)
        continue
    # Join in metadata columns
    result = result.merge(
        batch[['asset_id', 'region', 'sector', 'asset_type']],
        on='asset_id', how='left'
    )
    fetched_chunks.append(result)
    n_done += len(result)
    dt = time.time() - t0
    elapsed = time.time() - t_start
    eta_s = (elapsed / max(n_done, 1)) * (n_total - n_done)
    print(f'  batch {batch_idx+1:>4d}/{n_batches}  n={len(result):>4d}  '
          f'cum_done={n_done:>6d}/{n_total}  '
          f'batch_dt={dt:>5.1f}s  elapsed={elapsed/60:>5.1f}m  '
          f'eta~{eta_s/60:>5.1f}m')

    # Persist every 5 batches so a crash doesn't lose much work.
    if (batch_idx + 1) % 5 == 0 or (batch_idx + 1) == n_batches:
        new_df = pd.concat(fetched_chunks, ignore_index=True)
        if existing is not None:
            combined = pd.concat([existing, new_df], ignore_index=True)
        else:
            combined = new_df
        combined = combined.drop_duplicates(subset=['asset_id'], keep='last')
        # Reorder cols: metadata first, then A00..A63
        meta_cols = ['asset_id', 'region', 'sector', 'asset_type']
        combined = combined[meta_cols + AE_BAND_NAMES]
        combined.to_parquet(parquet_path, index=False)
        print(f'    [checkpoint] wrote {len(combined):,} rows to '
              f'{parquet_path.name}')

if errors_log:
    print(f'\nBatches with errors: {len(errors_log)}')
    for m in errors_log[-5:]:
        print(f'  {m}')
    print(f'  (showing last 5 of {len(errors_log)})')
print(f'\nStage 1 fetch complete in {(time.time() - t_start)/60:.1f} min')


AlphaEarth 2024 mosaic has 64 bands: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05']...
To fetch: 18,750  (skipping 0 already done)
  batch    1/38  n= 500  cum_done=   500/18750  batch_dt= 34.4s  elapsed=  0.6m  eta~ 20.9m
  batch    2/38  n= 500  cum_done=  1000/18750  batch_dt= 17.9s  elapsed=  0.9m  eta~ 15.5m
  batch    3/38  n= 500  cum_done=  1500/18750  batch_dt= 13.6s  elapsed=  1.1m  eta~ 12.6m
  batch    4/38  n= 500  cum_done=  2000/18750  batch_dt= 12.8s  elapsed=  1.3m  eta~ 11.0m
  batch    5/38  n= 500  cum_done=  2500/18750  batch_dt= 10.3s  elapsed=  1.5m  eta~  9.6m
    [checkpoint] wrote 2,500 rows to embeddings_2024.parquet
  batch    6/38  n= 500  cum_done=  3000/18750  batch_dt= 24.7s  elapsed=  1.9m  eta~ 10.0m
  batch    7/38  n= 500  cum_done=  3500/18750  batch_dt= 13.6s  elapsed=  2.1m  eta~  9.2m
  batch    8/38  n= 500  cum_done=  4000/18750  batch_dt= 13.4s  elapsed=  2.3m  eta~  8.7m
  batch    9/38  n= 500  cum_done=  4500/18750  batch_dt= 13.9s  elapsed=  2

In [14]:
import pandas as pd
import numpy as np

embeddings_df = pd.read_parquet(EMBEDDINGS_PARQUET)
print(f'Parquet shape: {embeddings_df.shape}  '
      f'(expected rows ~= {len(assets_df):,}, cols = 4 metadata + 64 features)')

# NaN check across the 64 embedding columns
emb_arr = embeddings_df[AE_BAND_NAMES].to_numpy(dtype=np.float64)
n_rows  = emb_arr.shape[0]
finite_mask = np.isfinite(emb_arr).all(axis=1)
n_finite = int(finite_mask.sum())
n_bad    = n_rows - n_finite
print(f'Finite-embedding rows: {n_finite:,} / {n_rows:,}  (bad: {n_bad:,})')

if n_bad > 0:
    bad_ids = embeddings_df.loc[~finite_mask, 'asset_id'].head(20).tolist()
    print(f'  First 20 bad asset_ids: {bad_ids}')
    print(f'  Dropping {n_bad:,} bad rows from the training set.')
    embeddings_df = embeddings_df[finite_mask].reset_index(drop=True)

# Distribution sanity check
print(f'\nEmbedding value range (all 64 bands, all rows):')
print(f'  min:    {emb_arr[finite_mask].min():.4f}')
print(f'  max:    {emb_arr[finite_mask].max():.4f}')
print(f'  mean:   {emb_arr[finite_mask].mean():.4f}')
print(f'  std:    {emb_arr[finite_mask].std():.4f}')

# Coverage check vs assets_df
missing = set(assets_df['asset_id'].astype(str)) - set(embeddings_df['asset_id'].astype(str))
if missing:
    print(f'\nWARNING: {len(missing):,} asset_ids in assets_df have NO embedding. '
          f'Re-run cell 7 to fetch them (it skips already-done ids).')
else:
    print(f'\nFull coverage: every asset in assets_df has an embedding.')

# Per-region and per-class counts in the final embedding set
print('\nPer-region embedding counts:')
print(embeddings_df.groupby('region').size().to_string())
print('\nPer-class embedding counts:')
print(embeddings_df.groupby('asset_type').size().sort_values(ascending=False).to_string())


Parquet shape: (18750, 68)  (expected rows ~= 18,750, cols = 4 metadata + 64 features)
Finite-embedding rows: 18,749 / 18,750  (bad: 1)
  First 20 bad asset_ids: ['osm_way_1036487691']
  Dropping 1 bad rows from the training set.

Embedding value range (all 64 bands, all rows):
  min:    -0.4462
  max:    0.4516
  mean:   -0.0075
  std:    0.1135


Per-region embedding counts:
region
africa               2842
asia                 2765
australia-oceania    3014
central-america      2565
europe               1736
north-america        2823
south-america        3004

Per-class embedding counts:
asset_type
transport.train_station           5046
water.storage_tank                3993
energy.distribution.other         3264
water.wastewater.plant            1299
energy.distribution.substation     975
water.treatment.plant              895
transport.airport                  871
energy.transmission.substation     676
energy.generation.solar_farm       631
energy.generation.power_plant      545
t

In [15]:
# Stratified 70/15/15 split by mapped asset_type, matching CROMA / Prithvi / SatlasS1.
# Spec mentioned 75/15/10 hash-based but the actual CROMA notebook uses 70/15/15
# stratified — we follow the real cross-FM invariant.

import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Map asset_type -> integer label and keep only labelable rows.
embeddings_df = embeddings_df[embeddings_df['asset_type'].isin(CLASS_NAMES)].reset_index(drop=True)
embeddings_df['label'] = embeddings_df['asset_type'].map(CLASS_TO_IDX).astype(int)
print(f'After class filter: {len(embeddings_df):,} rows')


def stratified_split(df, train_frac=0.7, val_frac=0.15, seed=SEED):
    """Same logic as CROMA's stratified_split, but on a DataFrame."""
    rng = random.Random(seed)
    train, val, test = [], [], []
    for cls in sorted(df['label'].unique()):
        idxs = df.index[df['label'] == cls].tolist()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        train.extend(idxs[:n_tr])
        val.extend(idxs[n_tr:n_tr + n_va])
        test.extend(idxs[n_tr + n_va:])
    return train, val, test


train_idx, val_idx, test_idx = stratified_split(embeddings_df)
print(f'Splits: train={len(train_idx):,}  val={len(val_idx):,}  test={len(test_idx):,}')


class AEEmbeddingDataset(Dataset):
    """Tensor of 64-D embeddings + metadata for one row at a time."""
    def __init__(self, df, indices):
        self.df = df.iloc[indices].reset_index(drop=True)
        self._emb_mat = self.df[AE_BAND_NAMES].to_numpy(dtype=np.float32)
        self.labels   = self.df['label'].astype(int).tolist()
        self.regions  = self.df['region'].tolist()
        self.sectors  = self.df['sector'].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        return {
            'embedding': torch.from_numpy(self._emb_mat[i]),
            'label':     int(self.labels[i]),
            'region':    self.regions[i],
            'sector':    self.sectors[i],
            'asset_id':  str(self.df.iloc[i]['asset_id']),
        }


train_set = AEEmbeddingDataset(embeddings_df, train_idx)
val_set   = AEEmbeddingDataset(embeddings_df, val_idx)
test_set  = AEEmbeddingDataset(embeddings_df, test_idx)

# Class distribution per split — sanity check stratification
from collections import Counter
print('\nClass distribution per split:')
print(f'  {"class":<35s} {"train":>6s} {"val":>6s} {"test":>6s}')
for c, name in enumerate(CLASS_NAMES):
    tr_n = sum(1 for l in train_set.labels if l == c)
    va_n = sum(1 for l in val_set.labels   if l == c)
    te_n = sum(1 for l in test_set.labels  if l == c)
    print(f'  [{c:>2d}] {name:<30s} {tr_n:>6d} {va_n:>6d} {te_n:>6d}')


After class filter: 18,749 rows
Splits: train=13,118  val=2,804  test=2,827

Class distribution per split:
  class                                train    val   test
  [ 0] energy.transmission.substation    473    101    102
  [ 1] energy.distribution.substation    682    146    147
  [ 2] energy.distribution.other        2284    489    491
  [ 3] energy.generation.power_plant     381     81     83
  [ 4] energy.generation.solar_farm      441     94     96
  [ 5] energy.generation.wind_farm         7      1      3
  [ 6] water.wastewater.plant            909    194    196
  [ 7] water.treatment.plant             626    134    135
  [ 8] water.storage_tank               2795    598    600
  [ 9] transport.airport                 609    130    132
  [10] transport.train_station          3532    756    758
  [11] transport.port_terminal             7      1      3
  [12] telecom.data_center               372     79     81


In [16]:
from torch.optim import AdamW
from collections import Counter, defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import torch.nn as nn
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  (linear probe on 64-D embeddings, CPU is fine)')


class LinearProbeHead(nn.Module):
    """Single Linear: 64 -> 13. No backbone — AlphaEarth embeddings are
    precomputed and frozen by design."""
    NAME = 'alphaearth_v1'

    def __init__(self, in_dim=AE_BAND_COUNT, num_classes=len(CLASS_NAMES),
                 dropout=0.1):
        super().__init__()
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, num_classes),
        )
    def forward(self, x):
        return self.head(x)


def compute_class_weights(labels, max_weight=WEIGHT_CAP):
    counts = Counter(labels)
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'embedding': torch.stack([b['embedding'] for b in batch]),
        'label':     torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region':    [b['region'] for b in batch],
        'sector':    [b['sector'] for b in batch],
    }


# --- Corrected per-sector F1 ----------------------------------------------
# Macro-average of per-class F1s for the classes in that sector, computed on
# the FULL test set. NOT a macro-F1 restricted to in-sector samples (which is
# the v1 definition we deprecated).
SECTOR_TO_CLASS_INDICES = defaultdict(list)
for i, name in enumerate(CLASS_NAMES):
    SECTOR_TO_CLASS_INDICES[name.split('.')[0]].append(i)


def per_sector_corrected(y_true, y_pred, per_class_f1):
    """per_class_f1: list-of-13. Returns {sector: {n, macro_f1, classes}}."""
    y_true_set = list(y_true)
    out = {}
    for sector, class_idxs in SECTOR_TO_CLASS_INDICES.items():
        in_sector_truth = sum(1 for y in y_true_set if y in set(class_idxs))
        f1s = [per_class_f1[i] for i in class_idxs]
        out[sector] = {
            'n': int(in_sector_truth),
            'macro_f1': float(np.mean(f1s)) if f1s else 0.0,
            'classes': [CLASS_NAMES[i] for i in class_idxs],
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['embedding'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    result = {
        'acc':          float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1':     float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion':    confusion_matrix(all_labels, all_preds,
                                         labels=list(range(len(CLASS_NAMES)))).tolist(),
    }
    if return_breakdowns:
        # Per-region: standard macro-F1 of the in-region subset (unchanged).
        def grouped_macro_f1(group_vals):
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, group_vals):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region'] = grouped_macro_f1(all_regions)
        # Per-sector: CORRECTED v2 — average of per-class F1s on the full test set.
        result['per_sector']                   = per_sector_corrected(all_labels, all_preds, per_class_f1)
        result['per_sector_corrected']         = result['per_sector']
        result['per_sector_correction_note']   = (
            'per_sector = macro-average of per-class F1s for classes in that sector, '
            'computed on the FULL test set (v2 definition). NOT a macro-F1 restricted '
            'to in-sector samples.'
        )
    return result


def train_one_run(train_set, val_set, test_set, *,
                  num_epochs, batch_size, lr,
                  weight_decay=1e-4, num_workers=0,
                  run_name='alphaearth_v1_linear_probe'):
    head = LinearProbeHead().to(DEVICE)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, collate_fn=collate, pin_memory=False)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, collate_fn=collate, pin_memory=False)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, collate_fn=collate, pin_memory=False)

    weights = compute_class_weights(train_set.labels).to(DEVICE)
    print('  Class weights (capped at 10.0):')
    for c, w in enumerate(weights.cpu().numpy()):
        print(f'    [{c:>2d}] {CLASS_NAMES[c]:<34s} {w:.4f}')
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)

    ckpt_dir        = Path(OUTPUT_DIR) / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path  = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt_path = ckpt_dir / 'checkpoint_final.pt'

    history, best_val_f1, best_epoch = [], -1.0, -1
    for epoch in range(num_epochs):
        head.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            emb    = batch['embedding'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(head(emb), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * emb.size(0)
            n += emb.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(head, val_loader)
        history.append({
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'time_s': time.time() - t0,
        })
        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch  = epoch + 1
            marker = ' *'
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': head.state_dict(),
                'val_macro_f1': val['macro_f1'],
                'history': history,
            }, best_ckpt_path)
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    torch.save({
        'epoch': num_epochs,
        'model_state_dict': head.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
        'best_val_f1': best_val_f1,
        'best_epoch':  best_epoch,
    }, final_ckpt_path)

    # ============== BEST_CKPT_BEFORE_TEST =================================
    # Restore best-val weights before the held-out test pass — same as
    # Prithvi / CROMA notebooks; fixes the S1-era methodological gap.
    # =====================================================================
    if best_ckpt_path.exists():
        ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
        head.load_state_dict(ckpt['model_state_dict'])
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
        print(f'\n  [BEST_CKPT_BEFORE_TEST] restored best-val state from '
              f'epoch {ckpt["epoch"]} (val_f1={ckpt["val_macro_f1"]:.4f}) before test')
    else:
        tested_with = 'final-epoch state (no best ckpt found)'
        print('\n  WARNING: no best-val ckpt — test runs on final-epoch state')

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(head, test_loader, return_breakdowns=True)
    return {
        'run_name':     run_name,
        'backbone':     'alphaearth_v1',
        'condition':    'precomputed_embeddings_linear_probe',
        'num_epochs':   num_epochs,
        'best_val_f1':  best_val_f1,
        'best_epoch':   best_epoch,
        'tail_mean_f1': float(np.mean(tail)),
        'tail_std_f1':  float(np.std(tail)),
        'history':      history,
        'test':         test,
        'tested_with':  tested_with,
    }


print('Training infrastructure ready (with best-val ckpt restore + corrected per-sector F1).')


Device: cuda  (linear probe on 64-D embeddings, CPU is fine)
Training infrastructure ready (with best-val ckpt restore + corrected per-sector F1).


In [18]:
# ============================================================================
# Optional smoke check. Default SMOKE_ONLY=True so the invocation cell below
# exits before training (training on 64-D embeddings is fast enough — minutes
# on CPU — that you may want to skip the smoke and just train).
# ============================================================================
SMOKE_ONLY = False

head = LinearProbeHead().to(DEVICE)
n_params = sum(p.numel() for p in head.parameters())
print(f'Linear probe params: {n_params:,}  (64 weights * 13 + 13 bias = '
      f'{AE_BAND_COUNT * len(CLASS_NAMES) + len(CLASS_NAMES)} expected)')

# Pull one batch
smoke_loader = DataLoader(train_set, batch_size=4, shuffle=False,
                          num_workers=0, collate_fn=collate)
batch = next(iter(smoke_loader))
emb = batch['embedding']
print(f'  Batch embedding shape: {tuple(emb.shape)}  (expect [4, 64])')
print(f'  Batch embedding range: [{emb.min().item():.4f}, {emb.max().item():.4f}]')

with torch.no_grad():
    logits = head(emb.to(DEVICE))
print(f'  Logits shape:          {tuple(logits.shape)}  (expect [4, {len(CLASS_NAMES)}])')
print(f'  Logits range:          [{logits.min().item():.4f}, {logits.max().item():.4f}]')
print(f'  All finite:            {torch.isfinite(logits).all().item()}')

# One train step
weights = compute_class_weights(train_set.labels).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW(head.parameters(), lr=LP_LR, weight_decay=1e-4)
optimizer.zero_grad()
loss = criterion(head(emb.to(DEVICE)), batch['label'].to(DEVICE))
loss.backward()
optimizer.step()
print(f'  Train step loss:       {loss.item():.4f}')

print(f'\nSmoke check PASSED.  SMOKE_ONLY = {SMOKE_ONLY}  — invocation cell will '
      f'{"NOT train" if SMOKE_ONLY else "TRAIN"}.')


Linear probe params: 845  (64 weights * 13 + 13 bias = 845 expected)
  Batch embedding shape: (4, 64)  (expect [4, 64])
  Batch embedding range: [-0.3581, 0.2976]
  Logits shape:          (4, 13)  (expect [4, 13])
  Logits range:          [-0.2111, 0.2296]
  All finite:            True
  Train step loss:       2.5853

Smoke check PASSED.  SMOKE_ONLY = False  — invocation cell will TRAIN.


In [19]:
# ============================================================================
# Train the linear probe.
# ============================================================================
import json as _json

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping training. Set SMOKE_ONLY=False in the '
          'cell above, re-run that cell and then this one to train.')
else:
    print('=' * 70)
    print('AlphaEarth — Linear Probe on precomputed 2024 embeddings')
    print(f'  REGIONS = {len(REGIONS)}, classes = {len(CLASS_NAMES)}, '
          f'epochs = {LP_EPOCHS}, batch = {LP_BATCH}, lr = {LP_LR}')
    print('=' * 70)

    results = {}
    results['linear_probe'] = train_one_run(
        train_set=train_set, val_set=val_set, test_set=test_set,
        num_epochs=LP_EPOCHS,
        batch_size=LP_BATCH,
        lr=LP_LR,
        run_name='alphaearth_v1_linear_probe',
    )

    out_path = Path(OUTPUT_DIR) / 'alphaearth_v1_linear_probe_results.json'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w') as f:
        _json.dump(results, f, indent=2)
    print(f'\nResults saved to: {out_path}')

    lp = results['linear_probe']
    print(f'\n=== AlphaEarth Linear Probe Summary ===')
    print(f'Tested with:     {lp["tested_with"]}')
    print(f'Best val F1:     {lp["best_val_f1"]:.4f}  (epoch {lp["best_epoch"]})')
    print(f'Tail mean F1:    {lp["tail_mean_f1"]:.4f} +/- {lp["tail_std_f1"]:.4f}')
    print(f'Test macro F1:   {lp["test"]["macro_f1"]:.4f}')
    print(f'Test accuracy:   {lp["test"]["acc"]:.4f}')

    print(f'\nPer-class F1 (test):')
    for c, name in enumerate(CLASS_NAMES):
        print(f'  [{c:>2d}] {name:<34s} {lp["test"]["per_class_f1"][c]:.4f}')

    print(f'\nPer-sector F1 (CORRECTED v2 — avg per-class F1, full test set):')
    for sector, stats in sorted(lp['test']['per_sector'].items()):
        print(f'  {sector:<10s} n={stats["n"]:>5d}  F1={stats["macro_f1"]:.4f}  '
              f'classes={len(stats["classes"])}')

    print(f'\nPer-region F1 (in-region subset macro-F1):')
    for region, stats in sorted(lp['test']['per_region'].items()):
        print(f'  {region:<22s} n={stats["n"]:>5d}  F1={stats["macro_f1"]:.4f}  acc={stats["acc"]:.4f}')


AlphaEarth — Linear Probe on precomputed 2024 embeddings
  REGIONS = 7, classes = 13, epochs = 25, batch = 16, lr = 0.001
  Class weights (capped at 10.0):
    [ 0] energy.transmission.substation     2.1334
    [ 1] energy.distribution.substation     1.4796
    [ 2] energy.distribution.other          0.4418
    [ 3] energy.generation.power_plant      2.6485
    [ 4] energy.generation.solar_farm       2.2882
    [ 5] energy.generation.wind_farm        10.0000
    [ 6] water.wastewater.plant             1.1101
    [ 7] water.treatment.plant              1.6119
    [ 8] water.storage_tank                 0.3610
    [ 9] transport.airport                  1.6569
    [10] transport.train_station            0.2857
    [11] transport.port_terminal            10.0000
    [12] telecom.data_center                2.7126
  ep   1  loss=2.3906  val_acc=0.3149  val_f1=0.1672 *
  ep   2  loss=2.2130  val_acc=0.3210  val_f1=0.2054 *
  ep   3  loss=2.1232  val_acc=0.3174  val_f1=0.2147 *
  ep   4  loss

In [24]:
import zipfile
import json
from pathlib import Path

# Pick one zip and look at its manifest
zip_path = Path('/content/drive/MyDrive/infra_fm/datasets/dataset_north-america_energy_v1_1k.zip')

with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
    # Find the manifest
    manifest_member = next((n for n in names if n.endswith('manifest.json')), None)
    if manifest_member:
        with zf.open(manifest_member) as f:
            manifest = json.load(f)
        n_records = len(manifest.get('records', []))
        n_npys = sum(1 for n in names if n.endswith('.npy'))
        print(f'Manifest records: {n_records}')
        print(f'Total .npy files in zip: {n_npys}')
        print(f'Match? {n_records == n_npys}')

Manifest records: 851
Total .npy files in zip: 851
Match? True


In [21]:
# ============================================================================
# Confusion matrix PNG + cross-FM comparison table.
# Runs only when training completed (SMOKE_ONLY=False).
# ============================================================================
if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping confusion-matrix render and FM comparison.')
else:
    import json as _json
    import numpy as np
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    # ---- Confusion matrix PNG ---------------------------------------------
    cm = np.array(lp['test']['confusion'])
    cm_norm = cm.astype(np.float64)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_norm, row_sums, out=np.zeros_like(cm_norm), where=row_sums > 0)

    short_names = [n.split('.', 1)[1] if '.' in n else n for n in CLASS_NAMES]
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(cm_norm, cmap='Greens', vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'AlphaEarth v1 linear probe — confusion (row-normalized)\n'
                 f'7 regions, 13 classes, test macro F1 = {lp["test"]["macro_f1"]:.3f}')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        color=color, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    cm_path = Path(OUTPUT_DIR) / 'confusion_matrix_alphaearth_7region.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Confusion matrix saved: {cm_path}')

    # ---- Cross-FM comparison ---------------------------------------------
    def _load_fm_result(path, label):
        try:
            with open(path) as f:
                d = _json.load(f)
            return d.get('linear_probe', d)
        except Exception as e:
            print(f'  [skip] {label}: {e.__class__.__name__}: {e}')
            return None

    candidates = [
        ('AlphaEarth',      lp),
        ('CROMA_base',
         _load_fm_result(f'{DRIVE_ROOT}/results/fm_eval_croma_v1/'
                         f'croma_base_v1_linear_probe_results.json', 'CROMA')),
        ('SatlasPretrain S2 (capped 7-region)',
         _load_fm_result(f'{DRIVE_ROOT}/results/fm_eval_satlas_multisector_v1/'
                         f'satlas_s2_full7region_v1_linear_probe_capped_results.json',
                         'SatlasS2')),
    ]
    rows = [(name, r) for name, r in candidates if r is not None]

    print('\n' + '=' * 78)
    print('Cross-FM comparison (linear probe, 7-region, 13-class)')
    print('=' * 78)
    print(f'{"model":<42s} {"test_F1":>8s} {"test_acc":>9s} {"best_epoch":>11s}')
    print('-' * 78)
    for name, r in rows:
        f1 = r.get('test', {}).get('macro_f1', float('nan'))
        ac = r.get('test', {}).get('acc',      float('nan'))
        be = r.get('best_epoch', '-')
        print(f'{name:<42s} {f1:>8.4f} {ac:>9.4f} {str(be):>11s}')

    print('\nPer-sector F1 (corrected v2):')
    sectors_all = sorted({s for _, r in rows for s in r.get('test', {}).get('per_sector', {})})
    header = f'  {"sector":<10s} ' + ' '.join(f'{name[:18]:>18s}' for name, _ in rows)
    print(header)
    for sec in sectors_all:
        cells = []
        for _, r in rows:
            v = r.get('test', {}).get('per_sector', {}).get(sec, {})
            cells.append(f'{v.get("macro_f1", float("nan")):>18.4f}')
        print(f'  {sec:<10s} ' + ' '.join(cells))


Confusion matrix saved: /content/drive/MyDrive/infra_fm/results/fm_eval_alphaearth_v1/confusion_matrix_alphaearth_7region.png

Cross-FM comparison (linear probe, 7-region, 13-class)
model                                       test_F1  test_acc  best_epoch
------------------------------------------------------------------------------
AlphaEarth                                   0.2494    0.3410          23
CROMA_base                                   0.2654    0.3743          13
SatlasPretrain S2 (capped 7-region)          0.1981    0.2504           -

Per-sector F1 (corrected v2):
  sector             AlphaEarth         CROMA_base SatlasPretrain S2 
  energy                 0.1682             0.0762             0.0459
  telecom                0.3837             0.1347             0.1425
  transport              0.2990             0.1160             0.1395
  water                  0.3175             0.1290             0.1007


In [25]:
import json
import numpy as np
with open('/content/drive/MyDrive/infra_fm/results/fm_eval_croma_v1/croma_base_v1_linear_probe_results.json') as f:
    results = json.load(f)
cm = np.array(results['linear_probe']['test']['confusion'])
print(f'Total test samples (cm sum): {cm.sum()}')
print(f'Per-class test n (cm row sums):')
for c, name in enumerate(CLASS_NAMES):
    print(f'  [{c:>2d}] {name:<34s} {int(cm.sum(axis=1)[c]):>4d}')

Total test samples (cm sum): 2899
Per-class test n (cm row sums):
  [ 0] energy.transmission.substation      108
  [ 1] energy.distribution.substation      152
  [ 2] energy.distribution.other           496
  [ 3] energy.generation.power_plant        89
  [ 4] energy.generation.solar_farm        102
  [ 5] energy.generation.wind_farm           6
  [ 6] water.wastewater.plant              202
  [ 7] water.treatment.plant               141
  [ 8] water.storage_tank                  607
  [ 9] transport.airport                   138
  [10] transport.train_station             765
  [11] transport.port_terminal               6
  [12] telecom.data_center                  87


In [26]:
import json
import numpy as np

path = '/content/drive/MyDrive/infra_fm/results/fm_eval_alphaearth_v1/alphaearth_v1_linear_probe_results.json'
with open(path) as f:
    results = json.load(f)

# Adapt path based on actual schema
cm = np.array(results['linear_probe']['test']['confusion'])
print(f'AlphaEarth test set total: {cm.sum()}')
print(f'Expected (matching other FMs): 2899')
print(f'Match: {cm.sum() == 2899}')

AlphaEarth test set total: 2827
Expected (matching other FMs): 2899
Match: False


In [27]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Load AlphaEarth results and embeddings parquet
ae_results_path = '/content/drive/MyDrive/infra_fm/results/fm_eval_alphaearth_v1/alphaearth_v1_linear_probe_results.json'
ae_embeddings_path = '/content/drive/MyDrive/infra_fm/data/alphaearth/embeddings_2024.parquet'

with open(ae_results_path) as f:
    ae_results = json.load(f)

ae_emb = pd.read_parquet(ae_embeddings_path)
print(f'Total embeddings in parquet: {len(ae_emb)}')
print(f'Total expected from manifests: 18,756')
print(f'Difference: {18756 - len(ae_emb)}')

# Check if any embeddings have null values
band_cols = [c for c in ae_emb.columns if c.startswith('A')]
null_counts = ae_emb[band_cols].isnull().any(axis=1).sum()
print(f'Embeddings with any null A* values: {null_counts}')

# Per-region count
print(f'\nPer-region embedding counts:')
print(ae_emb.groupby('region').size().to_string())

# Per-sector count
print(f'\nPer-sector embedding counts:')
print(ae_emb.groupby('sector').size().to_string())

Total embeddings in parquet: 18750
Total expected from manifests: 18,756
Difference: 6
Embeddings with any null A* values: 1

Per-region embedding counts:
region
africa               2842
asia                 2765
australia-oceania    3014
central-america      2565
europe               1737
north-america        2823
south-america        3004

Per-sector embedding counts:
sector
energy       6103
telecom       532
transport    5928
water        6187


In [1]:
!nvidia-smi

Sat Jun 27 18:50:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   78C    P0             69W /   72W |    4136MiB /  23034MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----